In [88]:
import os
from pymongo import MongoClient, ASCENDING
import re
from collections import defaultdict
from pprint import pprint

In [89]:
code_block_root_directory = r"E:\wearable-2-dataset-code-extraction"

In [90]:
# server
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'evaluate-responsible-entity'

In [91]:
# Function get all sub-directory under root directory
def get_all_subdirectories_under_one_level(root_dir: str):
    """
    Get all subdirectories directly under the given root directory (one level only).
    Return a list of full paths.
    """
    return [
        os.path.join(root_dir, name)
        if not os.path.isabs(name) else name
        for name in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, name))
    ]
def get_app_package_name(path: str) -> str:
    """
    Return the last directory name from a full path.
    Example:
        E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote
        -> 'adarshurs.android.vlcmobileremote'
    """
    return os.path.basename(os.path.normpath(path))
def group_docs_by_tps(docs):
    """
    Gom nhóm các document có 'tps_name' trùng với thư mục giữa
    'rag-external-data\\' và '\\code-rag' trong 'source'.
    
    Trả về: dict dạng {
        "Google": ["implemented", "non-implemented", ...],
        "Facebook": ["implemented", ...]
    }
    """
    grouped = defaultdict(list)

    for doc in docs:
        source = doc.get("source", "")
        tps_name = doc.get("tps_name", "")
        label = doc.get("label", "")

        # Tìm phần nằm giữa rag-external-data\ và \code-rag
        match = re.search(r"rag-external-data\\([^\\]+)\\code-rag", source)
        if match:
            extracted_name = match.group(1)
            # So sánh không phân biệt hoa/thường
            if extracted_name.lower() == tps_name.lower():
                grouped[tps_name].append(label)

    return dict(grouped)
def summarize_sdk_implement_status(data_dict, app_package_name):
    """
    Kiểm tra nếu trong danh sách của mỗi SDK có ít nhất 1 'implemented' 
    thì kết quả là 'implemented', ngược lại là 'non-implemented'.
    
    Parameters:
        data_dict (dict): {tên_SDK: [list các label]}
        app_package_name (str): tên gói ứng dụng
        
    Returns:
        dict: {app_package_name: ..., SDK1: status1, SDK2: status2, ...}
    """
    result = {"app_package_name": app_package_name}

    for sdk_name, status_list in data_dict.items():
        if any(s.lower() == "implemented" for s in status_list):
            result[sdk_name] = "implemented"
        else:
            result[sdk_name] = "non-implemented"

    return result


In [92]:
def connect_mongodb(uri, db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    connection = db[collection_name]
    
    return connection
def find_by_app_package_name(collection, app_package_name: str):
    """
    Lấy tất cả document có app_package_name tương ứng.
    Trả về list các document (dạng dict).
    """
    query = {"app_package_name": app_package_name}
    results = list(collection.find(query))
    return results

In [93]:
mongodb_connection = connect_mongodb(mongoDB_uri, mongoDB_database, mongoDB_collection)
level_1_code_directory_arr = get_all_subdirectories_under_one_level(code_block_root_directory)

In [94]:
for a in range(1,len(level_1_code_directory_arr)):
    level_1_code_directory_path = level_1_code_directory_arr[a]
    app_package_name = get_app_package_name(level_1_code_directory_path)
    print("==================================="+app_package_name+"===================================")
    print("App package name: ",app_package_name)
    docs = find_by_app_package_name(mongodb_connection, app_package_name)
    print(f"🔎 Found {len(docs)} documents for app: {app_package_name}\n")
    result = group_docs_by_tps(docs)
    pprint(result)
    print(type(result))
    break

===================================app.EnForma.fitcoapp.net===================================
App package name:  app.EnForma.fitcoapp.net
🔎 Found 295 documents for app: app.EnForma.fitcoapp.net

{'Firebase': ['non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented',
              'non-implemented'],
 'Sentry': ['non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
            'non-implemented',
      